# Preprocessing — RNA-seq + Methylation
Converts raw files into h5ad format for Geneformer and MethylGPT.

**Runtime → Run all**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q anndata pandas numpy tqdm

In [ ]:
# UPDATE to your shared folder name
SHARED_FOLDER_NAME = "multiomics-project"

from pathlib import Path
DRIVE_ROOT    = Path(f"/content/drive/MyDrive/{SHARED_FOLDER_NAME}")
RAW_DIR       = DRIVE_ROOT / "data/raw"
PROCESSED_DIR = DRIVE_ROOT / "data/processed"
MANIFEST      = DRIVE_ROOT / "data/manifests/matched_samples.csv"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Manifest exists:", MANIFEST.exists())

In [ ]:
# Preprocess RNA-seq → tcga_rna_seq.h5ad
import pandas as pd
import numpy as np
import anndata as ad
from tqdm import tqdm

manifest   = pd.read_csv(MANIFEST)
data_list  = []
obs_list   = []
var_names  = None
gene_names = None
gene_types = None

print(f"Processing {len(manifest)} RNA-seq files...")
for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
    file_path = RAW_DIR / row["project"] / "rna_seq" / row["rna_file_name"]
    if not file_path.exists():
        print(f"Missing: {file_path}")
        continue
    df = pd.read_csv(file_path, sep="\t", skiprows=1)
    df = df[df["gene_id"].str.startswith("ENSG")]
    df["gene_id"] = df["gene_id"].str.split(".").str[0]
    df = df.groupby("gene_id").agg({"unstranded": "sum", "gene_name": "first", "gene_type": "first"})
    data_list.append(df["unstranded"].values)
    obs_list.append({"case_id": row["case_id"], "cancer_type": row["project"], "file_id": row["rna_file_id"]})
    if var_names is None:
        var_names  = df.index.tolist()
        gene_names = df["gene_name"].tolist()
        gene_types = df["gene_type"].tolist()

adata_rna = ad.AnnData(
    X   = np.array(data_list),
    obs = pd.DataFrame(obs_list),
    var = pd.DataFrame(index=var_names, data={"gene_name": gene_names, "gene_type": gene_types})
)
out = PROCESSED_DIR / "tcga_rna_seq.h5ad"
adata_rna.write_h5ad(out)
print(f"Saved: {out}")
print(f"Shape: {adata_rna.shape}")

In [ ]:
# Preprocess Methylation → tcga_methylation.h5ad
# First upload probe_ids_type3.csv to the processed folder in Drive
PROBE_LIST = PROCESSED_DIR / "probe_ids_type3.csv"
print("Probe list exists:", PROBE_LIST.exists())

probes_df     = pd.read_csv(PROBE_LIST)
target_probes = probes_df[probes_df.columns[0]].tolist()
print(f"Targeting {len(target_probes)} CpG probes")

data_list = []
obs_list  = []

print(f"Processing {len(manifest)} methylation files...")
for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
    file_path = RAW_DIR / row["project"] / "methylation" / row["meth_file_name"]
    if not file_path.exists():
        print(f"Missing: {file_path}")
        continue
    sample_df   = pd.read_csv(file_path, sep="\t", header=None, names=["probe", "beta"])
    sample_df["beta"] = pd.to_numeric(sample_df["beta"], errors="coerce")
    sample_df   = sample_df.set_index("probe")
    sample_betas = sample_df.reindex(target_probes)["beta"].values
    data_list.append(sample_betas)
    obs_list.append({"case_id": row["case_id"], "cancer_type": row["project"], "file_id": row["meth_file_id"]})

X = np.stack(data_list)
X = np.nan_to_num(X, nan=0.0)

adata_meth = ad.AnnData(
    X   = X,
    obs = pd.DataFrame(obs_list),
    var = pd.DataFrame(index=target_probes)
)
out = PROCESSED_DIR / "tcga_methylation.h5ad"
adata_meth.write_h5ad(out)
print(f"Saved: {out}")
print(f"Shape: {adata_meth.shape}")

In [ ]:
# Verify both files
import anndata as ad
rna  = ad.read_h5ad(PROCESSED_DIR / "tcga_rna_seq.h5ad")
meth = ad.read_h5ad(PROCESSED_DIR / "tcga_methylation.h5ad")
print(f"RNA-seq shape      : {rna.shape}")
print(f"Methylation shape  : {meth.shape}")
print(f"Cancer types       : {rna.obs['cancer_type'].value_counts().to_dict()}")